# Visualize a shape

Create and inspect one adaptively tiled single-grain shape.

In [ ]:
# Resolution and target tiles 
BASE_RESOLUTION = 14
TARGET_TILES = 2**12

#Shape examples:
#SHAPE_KIND, SHAPE_PARAMETERS = "sphere", {"radius_nm": 40}
SHAPE_KIND, SHAPE_PARAMETERS = "ellipsoid", {"semi_axes_nm": (50, 25, 25)}
#SHAPE_KIND, SHAPE_PARAMETERS = "cylinder", {"radius_nm": 30, "length_nm": 60, "axis": "z"}
# SHAPE_KIND, SHAPE_PARAMETERS = "hexagonal_prism", {"side_length_nm": 40, "height_nm": 40, "axis": "z", "rotation_degrees": 0}
# SHAPE_KIND, SHAPE_PARAMETERS = "box", {"dimensions_nm": (80, 60, 40)}

In [ ]:
from pathlib import Path
import sys

import numpy as np

SINGLE_GRAIN_DIR = Path.cwd()
if not (SINGLE_GRAIN_DIR / "utils").is_dir():
    SINGLE_GRAIN_DIR = Path("python/experiments/single_grain").resolve()
sys.path.insert(0, str(SINGLE_GRAIN_DIR))

from utils.geometry import AxisAlignedBox, Cylinder, Ellipsoid, HexagonalPrism, Sphere, fill_domain_mesh
from utils.geometry_plotting import plot_prism_mesh

NM = 1e-9
p = SHAPE_PARAMETERS
if SHAPE_KIND == "sphere":
    domain = Sphere(center=(0, 0, 0), radius=p["radius_nm"] * NM)
elif SHAPE_KIND == "ellipsoid":
    domain = Ellipsoid(center=(0, 0, 0), semi_axes=np.asarray(p["semi_axes_nm"]) * NM)
elif SHAPE_KIND == "cylinder":
    domain = Cylinder(center=(0, 0, 0), radius=p["radius_nm"] * NM, length=p["length_nm"] * NM, axis=p.get("axis", "z"))
elif SHAPE_KIND == "hexagonal_prism":
    domain = HexagonalPrism(center=(0, 0, 0), side_length=p["side_length_nm"] * NM, height=p["height_nm"] * NM, axis=p.get("axis", "z"), rotation_degrees=p.get("rotation_degrees", 0))
elif SHAPE_KIND == "box":
    domain = AxisAlignedBox(center=(0, 0, 0), dimensions=np.asarray(p["dimensions_nm"]) * NM)
else:
    raise ValueError("SHAPE_KIND must be sphere, ellipsoid, cylinder, hexagonal_prism, or box")

extent = domain.bounding_box[1] - domain.bounding_box[0]
h0 = np.max(extent) / BASE_RESOLUTION
mesh = fill_domain_mesh(
    domain, N_target=TARGET_TILES, h0=h0, h_min=h0 / 16, max_depth=8,
    eps=1e-18, queue_policy="breadth_first",
    complete_layers=True, target_lower_tolerance=0.10, grid_shifts="half_step",
)

volume_fill = mesh.represented_volume / domain.volume
print(f"Tiles used: {mesh.achieved_tiles:,} (target: {TARGET_TILES:,})")
print(f"Volume fill: {volume_fill:.2%}")

levels = np.unique(mesh.levels)
print(f"Refinement levels used: {len(levels)} ({levels.min()}-{levels.max()})")
print(f"{'Level':>5} {'Tiles':>10} {'Shape volume':>14}")
print("-" * 33)
cell_volumes = np.prod(mesh.dimensions, axis=1)
for level in levels:
    mask = mesh.levels == level
    level_volume_fraction = cell_volumes[mask].sum() / domain.volume
    print(f"{level:>5} {mask.sum():>10,} {level_volume_fraction:>13.2%}")


fig, ax = plot_prism_mesh(mesh, alpha=0.45, max_plot_tiles=6000)
ax.set_title(f"{SHAPE_KIND.replace('_', ' ').title()} | {mesh.achieved_tiles:,} tiles | {volume_fill:.2%} volume fill")
fig.tight_layout()